In [1]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, roc_auc_score

In [2]:
swell = pd.read_csv("data/swell/train.csv")
if os.path.exists("data/swell/test.csv"):
    swell = pd.concat([swell, pd.read_csv("data/swell/test.csv")], ignore_index=True)

swell["binary_label"] = (swell["condition"] == "time pressure").astype(int)
print(swell.shape, swell["binary_label"].value_counts())

(410322, 37) binary_label
0    339172
1     71150
Name: count, dtype: int64


In [3]:
wesad = pd.read_csv("outputs/wesad_raw_features.csv")
wesad["binary_label"] = (wesad["ground_truth"] == "STRESSED").astype(int)
print(wesad["binary_label"].value_counts())

binary_label
0    1498
1     641
Name: count, dtype: int64


In [4]:
X = swell[["HR", "RMSSD"]].values
y = swell["binary_label"].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = StandardScaler()
f1s = []
for tr, te in skf.split(X, y):
    Xtr, Xte = scaler.fit_transform(X[tr]), scaler.transform(X[te])
    clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    clf.fit(Xtr, y[tr])
    f1s.append(f1_score(y[te], clf.predict(Xte), zero_division=0))
print(f"SWELL internal (stratified, not subject-independent): F1={np.mean(f1s):.4f}")

SWELL internal (stratified, not subject-independent): F1=0.9241


In [8]:
def cross_test(train_df, train_feats, test_df, test_feats, name_a, name_b):
    # Scale EACH dataset independently to its own distribution — this is the
    # standard approach for cross-dataset physiological transfer, since raw
    # HR/HRV values aren't directly comparable across studies with different
    # recording protocols and window lengths.
    train_scaler = StandardScaler()
    test_scaler = StandardScaler()

    Xtr = train_scaler.fit_transform(train_df[train_feats].values)
    ytr = train_df["binary_label"].values
    Xte = test_scaler.fit_transform(test_df[test_feats].values)  # fit on test set's own distribution
    yte = test_df["binary_label"].values

    clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    clf.fit(Xtr, ytr)
    preds = clf.predict(Xte)
    probs = clf.predict_proba(Xte)[:, 1]
    f1 = f1_score(yte, preds, zero_division=0)
    auc = roc_auc_score(yte, probs)
    print(f"{name_a} -> {name_b}: F1={f1:.4f}, AUC={auc:.4f}")
    print(classification_report(yte, preds, zero_division=0))
    return {"direction": f"{name_a}->{name_b}", "f1": f1, "auc": auc}

r1 = cross_test(wesad, ["hr", "hrv"], swell, ["HR", "RMSSD"], "WESAD", "SWELL")
r2 = cross_test(swell, ["HR", "RMSSD"], wesad, ["hr", "hrv"], "SWELL", "WESAD")

pd.DataFrame([r1, r2]).to_csv("outputs/cross_dataset_results.csv", index=False)

WESAD -> SWELL: F1=0.1173, AUC=0.3267
              precision    recall  f1-score   support

           0       0.78      0.62      0.69    339172
           1       0.09      0.17      0.12     71150

    accuracy                           0.54    410322
   macro avg       0.44      0.40      0.41    410322
weighted avg       0.66      0.54      0.59    410322

SWELL -> WESAD: F1=0.1936, AUC=0.4132
              precision    recall  f1-score   support

           0       0.68      0.77      0.72      1498
           1       0.23      0.17      0.19       641

    accuracy                           0.59      2139
   macro avg       0.46      0.47      0.46      2139
weighted avg       0.55      0.59      0.56      2139



In [7]:
print("WESAD hr/hrv stats:")
print(wesad[["hr", "hrv"]].describe())

print("\nSWELL HR/RMSSD stats:")
print(swell[["HR", "RMSSD"]].describe())

WESAD hr/hrv stats:
                hr          hrv
count  2139.000000  2139.000000
mean     79.843852    50.083776
std      18.358246    35.183823
min      48.000000     2.041241
25%      68.000000    26.672337
50%      76.000000    41.000871
75%      86.000000    62.818591
max     148.000000   218.909830

SWELL HR/RMSSD stats:
                  HR          RMSSD
count  410322.000000  410322.000000
mean       73.937573      14.978707
std        10.335769       4.122602
min        48.736986       5.529742
25%        66.707281      11.831264
50%        74.217815      14.417202
75%        80.330105      17.931444
max       113.754253      26.629477


In [9]:
print("WESAD — mean HR/HRV by class:")
print(wesad.groupby("binary_label")[["hr", "hrv"]].mean())

print("\nSWELL — mean HR/RMSSD by class:")
print(swell.groupby("binary_label")[["HR", "RMSSD"]].mean())

WESAD — mean HR/HRV by class:
                     hr        hrv
binary_label                      
0             72.312417  53.106986
1             97.444618  43.018615

SWELL — mean HR/RMSSD by class:
                     HR      RMSSD
binary_label                      
0             75.234247  14.932141
1             67.756333  15.200686
